In [8]:
import cmath
import math
import numpy as np

# ==========================================
# MDC Hardware Components
# ==========================================
class FIFO:
    """A cycle-accurate shift register (Delay Line)."""
    def __init__(self, depth):
        self.depth = depth
        self.buffer = [0j] * depth 
        
    def step(self, val):
        if self.depth == 0:
            return val
        out = self.buffer.pop(0)
        self.buffer.append(val)
        return out

class MDC_Commutator:
    """A combined pre-delay, switch, and post-delay alignment block."""
    def __init__(self, L):
        self.L = L
        # 1. Pre-Commutator Delay Lines (Staggering)
        self.pre = [FIFO(0), FIFO(L), FIFO(2 * L), FIFO(3 * L)]
        # 2. Post-Commutator Delay Lines (Alignment)
        self.post = [FIFO(3 * L), FIFO(2 * L), FIFO(L), FIFO(0)]
        
        # State variables for logging
        self.pre_out = [0j] * 4
        self.switch_out = [0j] * 4

    def step(self, inputs, clk):
        # Pass through Pre-FIFOs
        self.pre_out = [self.pre[i].step(inputs[i]) for i in range(4)]
        
        # Rotary Switch
        state = (clk // self.L) % 4
        for i in range(4):
            self.switch_out[i] = self.pre_out[(state - i) % 4]
            
        # Pass through Post-FIFOs
        post_out = [self.post[i].step(self.switch_out[i]) for i in range(4)]
        return post_out

# ==========================================
# Helper Functions
# ==========================================
def digit_reverse_128(index):
    stage1 = (index >> 5) & 0x3 
    stage2 = (index >> 3) & 0x3 
    stage3 = (index >> 1) & 0x3 
    stage4 = index & 0x1        
    return (stage4 << 6) | (stage3 << 4) | (stage2 << 2) | stage1

D_BITS = 18; D_FRAC = 17
T_BITS = 20; T_FRAC = 17

def to_scaled_int(val, bits, frac_bits):
    """Converts a float to a quantized, two's complement integer."""
    if abs(val) < 1e-10: val = 0.0
    scaled = math.floor(val * (1 << frac_bits))
    max_val = (1 << (bits - 1)) - 1
    min_val = -(1 << (bits - 1))
    
    if scaled > max_val: scaled = max_val
    elif scaled < min_val: scaled = min_val
        
    mask = (1 << bits) - 1
    return scaled & mask

def to_hex(val, bits, frac_bits):
    """Used primarily to format the single pure real/imag random inputs to mem files."""
    scaled = to_scaled_int(val, bits, frac_bits)
    hex_chars = (bits + 3) // 4
    return f"0x{scaled:0{hex_chars}X}"

def hex_cpx(c_val, bits, frac_bits):
    """Combines imaginary and real parts into a single 36-bit hex string."""
    real_int = to_scaled_int(c_val.real, bits, frac_bits)
    imag_int = to_scaled_int(c_val.imag, bits, frac_bits)
    
    # Pack Imaginary in upper bits, Real in lower bits
    combined = (imag_int << bits) | real_int
    
    # Calculate exact hex chars needed (36 bits -> 9 hex chars)
    hex_chars = (bits * 2 + 3) // 4
    return f"0x{combined:0{hex_chars}X}"

def fmt_arr(arr):
    """Helper to format an array of complex numbers into combined hex strings."""
    return "[" + ", ".join([hex_cpx(x, D_BITS, D_FRAC) for x in arr]) + "]"

def w(n, k):
    return cmath.exp(-2j * cmath.pi * k / n)

def radix4_butterfly(a, b, c, d, w1, w2, w3):
    s0 = a + c; s1 = a - c; s2 = b + d; s3 = b - d
    A_pre = s0 + s2; B_pre = s1 - 1j * s3; C_pre = s0 - s2; D_pre = s1 + 1j * s3
    return A_pre, B_pre * w1, C_pre * w2, D_pre * w3

def radix2_butterfly(a, b):
    return a + b, a - b

# ==========================================
# Cycle-Accurate MDC Pipeline Simulation
# ==========================================
def simulate_mdc_pipeline(input_data):
    final_output = [0j] * 128
    
    com1 = MDC_Commutator(L=8)
    com2 = MDC_Commutator(L=2)
    s4_delay = [0j] * 4 
    
    print("\n" + "="*95)
    print(" CYCLE-ACCURATE MDC PIPELINE SIMULATION (With FIFOs)")
    print("="*95)

    for clk in range(62):
        print(f"\n--- Clock {clk:02d} ---")
        
        # ---------------------------------------------------------
        # STAGE 1
        # ---------------------------------------------------------
        if clk < 32:
            a, b, c, d = input_data[clk], input_data[clk+32], input_data[clk+64], input_data[clk+96]
            w1, w2, w3 = w(128, clk), w(128, 2*clk), w(128, 3*clk)
            A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
            s1_out = [A/4, B/4, C/4, D/4]
        else:
            s1_out = [0j, 0j, 0j, 0j]

        # ---------------------------------------------------------
        # COMMUTATOR 1 (L=8)
        # ---------------------------------------------------------
        c1_out = com1.step(s1_out, clk)

        # Print the detailed trace for Commutator 1 with new 36-bit hex format
        print("  [Com1 Trace]")
        print(f"    Input (S1 Out) : {fmt_arr(s1_out)}")
        print(f"    Pre-FIFO Out   : {fmt_arr(com1.pre_out)}")
        print(f"    Post-FIFO In   : {fmt_arr(com1.switch_out)}")
        print(f"    Output (S2 In) : {fmt_arr(c1_out)}")

        # ---------------------------------------------------------
        # STAGE 2
        # ---------------------------------------------------------
        if 24 <= clk < 56:
            i = (clk - 24) % 8
            w1, w2, w3 = w(32, i), w(32, 2*i), w(32, 3*i)
            A, B, C, D = radix4_butterfly(*c1_out, w1, w2, w3)
            s2_out = [A/4, B/4, C/4, D/4]
        else:
            s2_out = [0j, 0j, 0j, 0j]

        # ---------------------------------------------------------
        # COMMUTATOR 2 (L=2)
        # ---------------------------------------------------------
        c2_out = com2.step(s2_out, clk)

        # ---------------------------------------------------------
        # STAGE 3
        # ---------------------------------------------------------
        if 30 <= clk < 62:
            i = (clk - 30) % 2
            w1, w2, w3 = w(8, i), w(8, 2*i), w(8, 3*i)
            A, B, C, D = radix4_butterfly(*c2_out, w1, w2, w3)
            s3_out = [A/4, B/4, C/4, D/4]
        else:
            s3_out = [0j, 0j, 0j, 0j]

        # ---------------------------------------------------------
        # STAGE 4 (Radix-2 Finisher & Final Alignment)
        # ---------------------------------------------------------
        if 30 <= clk < 62:
            local_clk = clk - 30
            if local_clk % 2 == 0:
                s4_delay = s3_out
            else:
                s3_block = local_clk // 2
                
                for wire in range(4):
                    a_s4 = s4_delay[wire]
                    b_s4 = s3_out[wire]
                    A, B = radix2_butterfly(a_s4, b_s4)
                    
                    A_hw, B_hw = A/2, B/2
                    
                    offset = (s3_block * 4 + wire) * 2
                    final_output[offset] = A_hw
                    final_output[offset + 1] = B_hw

    return final_output

# ==========================================
# Execution: Random Generation & Export
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)
    random_signal = np.random.uniform(-0.99, 0.99, 128) + 0j
    
    with open("input_signal.mem", "w") as f:
        for val in random_signal:
            # We pad original real-only inputs as 18-bit since imaginary is 0
            hex_str = to_hex(val.real, D_BITS, D_FRAC).replace("0x", "")
            f.write(f"{hex_str}\n")
    print("Generated 'input_signal.mem' with 128 random samples.")

    hw_result_raw = simulate_mdc_pipeline(list(random_signal))
    
    hw_result = [0j] * 128
    for j in range(128):
        hw_result[digit_reverse_128(j)] = hw_result_raw[j]
        
    ideal_result = np.fft.fft(random_signal)
    
    hw_result_unscaled = np.array(hw_result) * 128.0
    error = np.max(np.abs(hw_result_unscaled - ideal_result))
    
    signal_power = np.sum(np.abs(ideal_result)**2)
    noise_power = np.sum(np.abs(hw_result_unscaled - ideal_result)**2)
    sqnr = 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')
    
    print("\n===================================================")
    print("             DSP Verification Metrics              ")
    print("===================================================")
    print(f"Mathematical Error : {error:.4e}")
    print(f"Pipeline SQNR      : {sqnr:.2f} dB")
    print("===================================================")

Generated 'input_signal.mem' with 128 random samples.

 CYCLE-ACCURATE MDC PIPELINE SIMULATION (With FIFOs)

--- Clock 00 ---
  [Com1 Trace]
    Input (S1 Out) : [0x000034036, 0x1CFF817B9, 0x000001127, 0xE300417B9]
    Pre-FIFO Out   : [0x000034036, 0x000000000, 0x000000000, 0x000000000]
    Post-FIFO In   : [0x000034036, 0x000000000, 0x000000000, 0x000000000]
    Output (S2 In) : [0x000000000, 0x000000000, 0x000000000, 0x000000000]

--- Clock 01 ---
  [Com1 Trace]
    Input (S1 Out) : [0x00000DC73, 0xDDBD060CC, 0xFF4601D81, 0x1CE1879AD]
    Pre-FIFO Out   : [0x00000DC73, 0x000000000, 0x000000000, 0x000000000]
    Post-FIFO In   : [0x00000DC73, 0x000000000, 0x000000000, 0x000000000]
    Output (S2 In) : [0x000000000, 0x000000000, 0x000000000, 0x000000000]

--- Clock 02 ---
  [Com1 Trace]
    Input (S1 Out) : [0x00003DD86, 0xC10B47DB9, 0x0175FE2A2, 0x2E228D485]
    Pre-FIFO Out   : [0x00003DD86, 0x000000000, 0x000000000, 0x000000000]
    Post-FIFO In   : [0x00003DD86, 0x000000000, 0x000